# Imports and configs

In [ ]:
import sys
import pandas as pd
from PIL import Image
from pathlib import Path
from clustimage import Clustimage

sys.path.insert(0, "/workspaces/WebIdentification/src")

In [ ]:
DATA_DIR = Path("/workspaces/WebIdentification/notebooks/clustimage_images")
RUN_DATA_EXPORT = False
MAX_IMAGES = 100
RANDOM_STATE = 42

# Get the data from the database

In [ ]:
from webidentification.pipeline.export_clustimage_images import main as export_images

if RUN_DATA_EXPORT:
    export_images()

metadata = pd.read_csv(DATA_DIR / "metadata.csv")
metadata["filepath"] = metadata["image"].apply(lambda p: str(DATA_DIR / p))
metadata = metadata[metadata["filepath"].apply(lambda p: Path(p).exists())].copy()

# Keep notebook memory usage manageable.
if len(metadata) > MAX_IMAGES:
    metadata = metadata.sample(n=MAX_IMAGES, random_state=RANDOM_STATE).reset_index(drop=True)

# Drop problematic/degenerate images that can cause NaN preprocessing.
def is_valid_image(path: str) -> bool:
    try:
        with Image.open(path) as img:
            return img.width > 1 and img.height > 1
    except Exception:
        return False

metadata = metadata[metadata["filepath"].apply(is_valid_image)].reset_index(drop=True)
print(f"Images selected for clustering: {len(metadata)}")
metadata.head()

# Cluster

In [ ]:
cl = Clustimage(method="hog")
results = cl.fit_transform(metadata["filepath"].tolist())
results

## Plotting by Domain and Website
You can use the `labels` argument in the plotting functions to color/group by domain or website.

In [ ]:
cl.plot(labels=metadata["website"])
cl.scatter()


In [ ]:
cl.plot(labels=metadata["domain"])
cl.scatter()
cl.plot_unique()


In [ ]:
results